# FASE 1: Exploração e Preparação de Dados (EDA)

## Objetivo
Entender os dados, realizar análise exploratória e preparar o dataset limpo para modelagem com Thompson Sampling.

## 📊 Base de Dados

Este notebook baixa e processa a base **bank-marketing** automaticamente:

| Base | Autor | Descrição | Registros |
|------|-------|-----------|----------|
| **bank-marketing** | henriqueyamahata | Campanhas bancárias, propensão de conversão | ~41k |

**Sistema de Cache:** Se os dados já existem, não baixa novamente! 🚀

---

## 1️⃣ Setup e Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
import warnings
import os
import subprocess
from pathlib import Path
warnings.filterwarnings('ignore')

# Configuração de visualização
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports realizados com sucesso!")

ModuleNotFoundError: No module named 'pandas'

## 2️⃣ Configuração de Datasets e Cache

In [ ]:
# Configuração da base selecionada
DATASET_CONFIG = {
    'bank-marketing': {
        'name': '1️⃣ Bank Marketing (henriqueyamahata)',
        'kaggle_id': 'henriqueyamahata/bank-marketing',
        'description': 'Campanhas bancárias, propensão de conversão e decisão de oferta',
        'separator': ';',
        'target_col': 'y',
        'size_mb': '~0.4 MB'
    }
}

RAW_DATA_DIR = Path('../data/raw')
PROCESSED_DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')

# Criar pastas
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Configuração do dataset inicializada")
print(f"\n📁 Diretórios:")
print(f"  • Raw: {RAW_DATA_DIR}")
print(f"  • Processed: {PROCESSED_DATA_DIR}")
print(f"  • Models: {MODELS_DIR}")

✅ Configuração de datasets inicializada

📁 Diretórios:
  • Raw: ../data/raw
  • Processed: ../data/processed
  • Models: ../models


## 3️⃣ Sistema de Cache - Download Automático com Detecção

In [ ]:
def dataset_exists(dataset_name):
    """Verifica se dataset já foi baixado (cache)"""
    dataset_dir = RAW_DATA_DIR / dataset_name
    if not dataset_dir.exists():
        return False
    
    # Verificar se tem arquivos CSV
    csv_files = list(dataset_dir.glob('*.csv'))
    return len(csv_files) > 0

def download_dataset(dataset_name, config):
    """Baixa dataset do Kaggle com cache"""
    dataset_dir = RAW_DATA_DIR / dataset_name
    
    # Verificar cache
    if dataset_exists(dataset_name):
        print(f"✅ {config['name']}")
        print(f"   📦 Cache encontrado - usando dados locais")
        csv_files = list(dataset_dir.glob('*.csv'))
        print(f"   📁 Arquivos: {[f.name for f in csv_files]}")
        return True
    
    # Baixar
    print(f"⬇️  {config['name']}")
    print(f"   📥 Baixando do Kaggle...")
    
    try:
        dataset_dir.mkdir(parents=True, exist_ok=True)
        
        # Baixar dataset
        subprocess.run([
            'kaggle', 'datasets', 'download',
            '-d', config['kaggle_id'],
            '-p', str(dataset_dir)
        ], check=True, capture_output=True)
        
        # Descompactar
        zip_files = list(dataset_dir.glob('*.zip'))
        for zip_file in zip_files:
            subprocess.run(f'cd {dataset_dir} && unzip -q {zip_file.name} && rm {zip_file.name}', 
                           shell=True, check=True)
        
        print(f"   ✅ Download concluído")
        csv_files = list(dataset_dir.glob('*.csv'))
        print(f"   📁 Arquivos: {[f.name for f in csv_files]}")
        return True
    
    except Exception as e:
        print(f"   ❌ Erro: {e}")
        print(f"   💡 Baixe manualmente: https://www.kaggle.com/datasets/{config['kaggle_id']}")
        return False

# Resumo das bases
print("\n" + "="*70)
print("📊 SISTEMA DE CACHE - VERIFICANDO DATASETS")
print("="*70)

for dataset_name, config in DATASETS_CONFIG.items():
    if dataset_exists(dataset_name):
        print(f"\n✅ {config['name']} [{config['size_mb']}]")
        print(f"   🎯 Cache encontrado")
    else:
        print(f"\n⬇️  {config['name']} [{config['size_mb']}]")
        print(f"   ⏳ Fila de download")

print("\n" + "="*70)


📊 SISTEMA DE CACHE - VERIFICANDO DATASETS

✅ 1️⃣ Bank Marketing (henriqueyamahata) [~0.4 MB]
   🎯 Cache encontrado

✅ 2️⃣ Bank Marketing Data Set (hariharanpavan) [~1.2 MB]
   🎯 Cache encontrado

✅ 3️⃣ Bank Term Deposit Subscription (dharmik34) [~0.2 MB]
   🎯 Cache encontrado

⬇️  4️⃣ Telemarketing JYB Dataset (aguado) [~0.1 MB]
   ⏳ Fila de download



## 4️⃣ Executar Download com Cache

In [ ]:
# Baixar a base bank-marketing (usando cache se existir)
print("\n" + "="*70)
print("📥 INICIANDO DOWNLOAD (COM CACHE)")
print("="*70 + "\n")

download_status = {}
for dataset_name, config in DATASETS_CONFIG.items():
    success = download_dataset(dataset_name, config)
    download_status[dataset_name] = success
    print()

# Resumo
print("="*70)
print("📊 RESUMO DO DOWNLOAD")
print("="*70)
success_count = sum(download_status.values())
total_count = len(download_status)
print(f"\n✅ {success_count}/{total_count} base pronta")

if success_count == total_count:
    print("\n🎉 A base bank-marketing foi baixada com sucesso!")
else:
    print("\n⚠️  O download falhou. Baixe manualmente.")


📥 INICIANDO DOWNLOADS (COM CACHE)

✅ 1️⃣ Bank Marketing (henriqueyamahata)
   📦 Cache encontrado - usando dados locais
   📁 Arquivos: ['bank-additional-full.csv']

✅ 2️⃣ Bank Marketing Data Set (hariharanpavan)
   📦 Cache encontrado - usando dados locais
   📁 Arquivos: ['bank.csv', 'bank-full.csv']

✅ 3️⃣ Bank Term Deposit Subscription (dharmik34)
   📦 Cache encontrado - usando dados locais
   📁 Arquivos: ['bank.csv', 'bank-full.csv']

⬇️  4️⃣ Telemarketing JYB Dataset (aguado)
   📥 Baixando do Kaggle...
   ✅ Download concluído
   📁 Arquivos: ['test.csv', 'train.csv']

📊 RESUMO DE DOWNLOADS

✅ 4/4 bases prontas

🎉 Todas as 4 bases foram baixadas com sucesso!


## 5️⃣ Carregar o Dataset

In [ ]:
def load_dataset(dataset_name, config):
    """Carrega dataset com detecção automática de arquivo"""
    dataset_dir = RAW_DATA_DIR / dataset_name
    
    # Procurar arquivos CSV
    csv_files = list(dataset_dir.glob('*.csv'))
    
    if not csv_files:
        print(f"❌ Nenhum arquivo CSV encontrado em {dataset_dir}")
        return None
    
    # Tentar carregar o primeiro CSV encontrado
    csv_file = csv_files[0]
    
    try:
        df = pd.read_csv(str(csv_file), sep=config['separator'])
        return df
    except Exception as e:
        print(f"❌ Erro ao carregar: {e}")
        return None

# Carregar o dataset
print("\n" + "="*70)
print("📖 CARREGANDO DATASET")
print("="*70 + "\n")

datasets = {}
dataset_info_list = []

for dataset_name, config in DATASETS_CONFIG.items():
    df = load_dataset(dataset_name, config)
    
    if df is not None:
        datasets[dataset_name] = df
        
        # Armazenar info
        dataset_info_list.append({
            'Dataset': config['name'],
            'Registros': f"{df.shape[0]:,}",
            'Colunas': df.shape[1],
            'Target': config['target_col'],
            'Memoria': f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
        })
        
        print(f"✅ {config['name']}")
        print(f"   📊 Shape: {df.shape[0]:,} × {df.shape[1]}")
        print(f"   🎯 Target: {config['target_col']}\n")
    else:
        print(f"❌ {config['name']} - Falha ao carregar\n")

# Tabela do dataset
print("="*70)
print("📊 INFORMAÇÕES DO DATASET")
print("="*70)
df_info = pd.DataFrame(dataset_info_list)
print(df_info.to_string(index=False))
print(f"\n✅ {len(datasets)}/{len(DATASETS_CONFIG)} dataset carregado com sucesso")


📖 CARREGANDO DATASETS

✅ 1️⃣ Bank Marketing (henriqueyamahata)
   📊 Shape: 41,188 × 21
   🎯 Target: y

✅ 2️⃣ Bank Marketing Data Set (hariharanpavan)
   📊 Shape: 4,521 × 17
   🎯 Target: y

✅ 3️⃣ Bank Term Deposit Subscription (dharmik34)
   📊 Shape: 4,521 × 1
   🎯 Target: target

✅ 4️⃣ Telemarketing JYB Dataset (aguado)
   📊 Shape: 12,543 × 1
   🎯 Target: y

📊 COMPARAÇÃO DE DATASETS
                                       Dataset Registros  Colunas Target  Memoria
         1️⃣ Bank Marketing (henriqueyamahata)    41,188       21      y 26.80 MB
  2️⃣ Bank Marketing Data Set (hariharanpavan)     4,521       17      y  2.58 MB
3️⃣ Bank Term Deposit Subscription (dharmik34)     4,521        1 target  0.65 MB
        4️⃣ Telemarketing JYB Dataset (aguado)    12,543        1      y  1.98 MB

✅ 4/4 datasets carregados com sucesso


## 6️⃣ Exploração Rápida do Dataset

In [ ]:
# Análise rápida de cada dataset
for dataset_name, df in datasets.items():
    config = DATASETS_CONFIG[dataset_name]
    target_col = config['target_col']
    
    print(f"\n" + "="*70)
    print(f"🔍 EXPLORAÇÃO: {config['name']}")
    print("="*70)
    
    # Tipos de dados
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(include=['object']).columns
    
    print(f"\n📊 Tipos de dados:")
    print(f"  • Numéricas: {len(numeric_cols)}")
    print(f"  • Categóricas: {len(categorical_cols)}")
    
    # Valores faltantes
    missing = df.isnull().sum().sum()
    print(f"\n📌 Valores faltantes: {missing}")
    
    # Target
    if target_col in df.columns:
        target_counts = df[target_col].value_counts()
        conversion_rate = target_counts.iloc[0] / len(df) * 100 if len(target_counts) > 0 else 0
        print(f"\n🎯 Target ({target_col}):")
        print(f"  Distribuição: {dict(target_counts)}")
        print(f"  Proporção: {target_counts.values}")
    
    # Amostra
    print(f"\n📄 Primeiras 3 linhas:")
    print(df.head(3).to_string())


🔍 EXPLORAÇÃO: 1️⃣ Bank Marketing (henriqueyamahata)

📊 Tipos de dados:
  • Numéricas: 10
  • Categóricas: 11

📌 Valores faltantes: 0

🎯 Target (y):
  Distribuição: {'no': np.int64(36548), 'yes': np.int64(4640)}
  Proporção: [36548  4640]

📄 Primeiras 3 linhas:
   age        job  marital    education  default housing loan    contact month day_of_week  duration  campaign  pdays  previous     poutcome  emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  nr.employed   y
0   56  housemaid  married     basic.4y       no      no   no  telephone   may         mon       261         1    999         0  nonexistent           1.1          93.994          -36.4      4.857       5191.0  no
1   57   services  married  high.school  unknown      no   no  telephone   may         mon       149         1    999         0  nonexistent           1.1          93.994          -36.4      4.857       5191.0  no
2   37   services  married  high.school       no     yes   no  telephone   may         mon      

## 7️⃣ Processar Datasets Individualmente

⚠️ **Escolha qual dataset processar abaixo**

In [ ]:
# ==================== CONFIGURAÇÃO: DATASET SELECIONADO ====================

SELECTED_DATASET = 'bank-marketing'

# ===========================================================================

if SELECTED_DATASET not in datasets:
    print(f"❌ Dataset '{SELECTED_DATASET}' não foi carregado!")
    print(f"\nDatasets disponíveis: {list(datasets.keys())}")
else:
    config = DATASETS_CONFIG[SELECTED_DATASET]
    df = datasets[SELECTED_DATASET]
    target_col = config['target_col']
    
    print(f"✅ Dataset selecionado: {config['name']}")
    print(f"📝 Descrição: {config['description']}")
    print(f"📊 Dimensões: {df.shape}")
    print(f"🎯 Target: {target_col}")

✅ Dataset selecionado: 1️⃣ Bank Marketing (henriqueyamahata)
📝 Descrição: Campanhas bancárias, propensão de conversão e decisão de oferta
📊 Dimensões: (41188, 21)
🎯 Target: y


## 8️⃣ Processamento do Dataset Selecionado

In [ ]:
# Copiar para processamento
df_processed = df.copy()

print(f"\n" + "="*70)
print(f"🔄 PROCESSANDO: {config['name']}")
print("="*70)

# 1. Remover vazamento temporal
cols_to_remove = ['duration']
cols_to_remove = [col for col in cols_to_remove if col in df_processed.columns]

if cols_to_remove:
    df_processed = df_processed.drop(cols_to_remove, axis=1)
    print(f"\n✅ Removidas colunas de vazamento: {cols_to_remove}")
else:
    print(f"\n✅ Nenhuma coluna de vazamento para remover")

# 2. Tratar valores faltantes
missing_after = df_processed.isnull().sum()
if missing_after.sum() > 0:
    for col in missing_after[missing_after > 0].index:
        if df_processed[col].dtype in ['int64', 'float64']:
            df_processed[col].fillna(df_processed[col].median(), inplace=True)
        else:
            mode_val = df_processed[col].mode()
            if len(mode_val) > 0:
                df_processed[col].fillna(mode_val[0], inplace=True)
    print(f"✅ Valores faltantes tratados")
else:
    print(f"✅ Nenhum valor faltante")

# 3. Converter target para numérico
if target_col in df_processed.columns:
    if 'yes' in df_processed[target_col].values:
        df_processed[target_col] = (df_processed[target_col] == 'yes').astype(int)
    elif 'Yes' in df_processed[target_col].values:
        df_processed[target_col] = (df_processed[target_col] == 'Yes').astype(int)
    elif df_processed[target_col].dtype == 'object':
        df_processed[target_col] = (df_processed[target_col].astype(bool)).astype(int)
    print(f"✅ Target convertido para numérico")

# 4. Encoding categóricas
categorical_cols_encode = [
    col for col in df_processed.select_dtypes(include=['object']).columns
    if col != target_col
]

label_encoders = {}
if categorical_cols_encode:
    for col in categorical_cols_encode:
        le = LabelEncoder()
        df_processed[col] = le.fit_transform(df_processed[col].astype(str))
        label_encoders[col] = le
    print(f"✅ {len(categorical_cols_encode)} colunas categóricas encoded")

# 5. Separar e normalizar
X = df_processed.drop(target_col, axis=1)
y = df_processed[target_col]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"✅ Features normalizadas (média={X_scaled.mean().mean():.4f}, std={X_scaled.std().mean():.4f})")

# 6. Recombinar
data_final = X_scaled.copy()
data_final[target_col] = y.values

print(f"\n📊 Dataset final: {data_final.shape}")
print(f"🎯 Taxa de conversão: {(y == 1).sum() / len(y) * 100:.2f}%")


🔄 PROCESSANDO: 1️⃣ Bank Marketing (henriqueyamahata)

✅ Removidas colunas de vazamento: ['duration']
✅ Nenhum valor faltante
✅ Target convertido para numérico
✅ 10 colunas categóricas encoded
✅ Features normalizadas (média=-0.0000, std=1.0000)

📊 Dataset final: (41188, 20)
🎯 Taxa de conversão: 11.27%


## 9️⃣ Salvar Dataset Processado

In [ ]:
# Criar subpasta com nome do dataset
dataset_processed_dir = PROCESSED_DATA_DIR / SELECTED_DATASET
dataset_processed_dir.mkdir(parents=True, exist_ok=True)

# Salvar dados
data_final.to_csv(dataset_processed_dir / 'data_processed.csv', index=False)
data_final.to_parquet(dataset_processed_dir / 'data_processed.parquet', index=False)
X_scaled.to_csv(dataset_processed_dir / 'X_features.csv', index=False)
y.to_csv(dataset_processed_dir / 'y_target.csv', index=False, header=['y'])

# Salvar modelos
joblib.dump(scaler, MODELS_DIR / f'scaler_{SELECTED_DATASET}.pkl')
joblib.dump(label_encoders, MODELS_DIR / f'label_encoders_{SELECTED_DATASET}.pkl')

print(f"\n✅ Dados salvos em: {dataset_processed_dir}")
print(f"\n📁 Arquivos gerados:")
print(f"  • data_processed.csv")
print(f"  • data_processed.parquet")
print(f"  • X_features.csv")
print(f"  • y_target.csv")
print(f"\n🎛️  Modelos salvos em: {MODELS_DIR}")
print(f"  • scaler_{SELECTED_DATASET}.pkl")
print(f"  • label_encoders_{SELECTED_DATASET}.pkl")


✅ Dados salvos em: ../data/processed/bank-marketing

📁 Arquivos gerados:
  • data_processed.csv
  • data_processed.parquet
  • X_features.csv
  • y_target.csv

🎛️  Modelos salvos em: ../models
  • scaler_bank-marketing.pkl
  • label_encoders_bank-marketing.pkl


## 🔟 Criar Documentação do Dataset

In [ ]:
# Criar documentação para o dataset processado
dataset_info = f"""# {config['name']}

## 📊 Informações do Dataset

### Origem
- **Base Kaggle**: {config['kaggle_id']}
- **Descrição**: {config['description']}
- **Data de processamento**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

### Dimensões
- **Registros originais**: {df.shape[0]:,}
- **Registros processados**: {data_final.shape[0]:,}
- **Colunas originais**: {df.shape[1]}
- **Colunas finais**: {data_final.shape[1]}

## 🎯 Distribuição do Target
- **Classe 0 (Não-conversão)**: {(y == 0).sum():,} ({(y == 0).sum() / len(y) * 100:.2f}%)
- **Classe 1 (Conversão)**: {(y == 1).sum():,} ({(y == 1).sum() / len(y) * 100:.2f}%)
- **Taxa de conversão**: {(y == 1).sum() / len(y) * 100:.2f}%

## 🔄 Transformações Realizadas
- ✅ Vazamento temporal removido
- ✅ Valores faltantes tratados
- ✅ {len(label_encoders)} colunas categóricas encoded
- ✅ Features normalizadas (StandardScaler)

## 📁 Arquivos Gerados
- `data_processed.csv` — Dataset completo
- `data_processed.parquet` — Formato eficiente
- `X_features.csv` — Features apenas
- `y_target.csv` — Target apenas
- `scaler_{SELECTED_DATASET}.pkl` — StandardScaler
- `label_encoders_{SELECTED_DATASET}.pkl` — Encoders

## ✅ Pronto para Thompson Sampling!
"""

with open(dataset_processed_dir / 'dataset_info.md', 'w') as f:
    f.write(dataset_info)

print(f"\n✅ Documentação criada: dataset_info.md")


✅ Documentação criada: dataset_info.md


## 1️⃣1️⃣ Criar Documento do Dataset

In [ ]:
# Gerar tabela comparativa de todos os datasets
comparison_data = []

for dataset_name, df_temp in datasets.items():
    config_temp = DATASETS_CONFIG[dataset_name]
    target_col_temp = config_temp['target_col']
    
    if target_col_temp in df_temp.columns:
        if 'yes' in df_temp[target_col_temp].values:
            conversion = (df_temp[target_col_temp] == 'yes').sum() / len(df_temp) * 100
        elif 'Yes' in df_temp[target_col_temp].values:
            conversion = (df_temp[target_col_temp] == 'Yes').sum() / len(df_temp) * 100
        else:
            conversion = df_temp[target_col_temp].astype(bool).sum() / len(df_temp) * 100
    else:
        conversion = 0
    
    comparison_data.append({
        'Dataset': config_temp['name'].split(' ')[1],
        'Autor': config_temp['name'].split('(')[1].rstrip(')'),
        'Registros': f"{df_temp.shape[0]:,}",
        'Colunas': df_temp.shape[1],
        'Taxa Conversão': f"{conversion:.2f}%",
        'Separador': config_temp['separator'],
        'Target': target_col_temp
    })

df_comparison = pd.DataFrame(comparison_data)

print(f"\n" + "="*70)
print("📊 COMPARAÇÃO DE TODOS OS DATASETS")
print("="*70)
print(df_comparison.to_string(index=False))

# Salvar tabela comparativa
comparison_file = PROCESSED_DATA_DIR / 'DATASETS_COMPARISON.csv'
df_comparison.to_csv(comparison_file, index=False)
print(f"\n✅ Tabela comparativa salva: {comparison_file.name}")


📊 COMPARAÇÃO DE TODOS OS DATASETS
      Dataset            Autor Registros  Colunas Taxa Conversão Separador Target
         Bank henriqueyamahata    41,188       21         11.27%         ;      y
         Bank   hariharanpavan     4,521       17         11.52%         ,      y
         Bank        dharmik34     4,521        1          0.00%         , target
Telemarketing           aguado    12,543        1          0.00%         ,      y

✅ Tabela comparativa salva: DATASETS_COMPARISON.csv


## 1️⃣2️⃣ Resumo Final e Próximas Etapas

In [ ]:
print(f"\n" + "="*70)
print("✅ FASE 1 CONCLUÍDA - RESUMO FINAL")
print("="*70)

print(f"\n📥 Download com Cache:")
print(f"  ✅ {sum(download_status.values())}/{len(DATASETS_CONFIG)} base baixada")

print(f"\n📖 Dataset Carregado:")
for dataset_name in datasets.keys():
    print(f"  ✅ {DATASETS_CONFIG[dataset_name]['name']}")

print(f"\n🔄 Dataset Processado:")
print(f"  📊 {config['name']}")
print(f"  📁 Salvo em: data/processed/{SELECTED_DATASET}/")
print(f"  💾 {data_final.shape[0]:,} × {data_final.shape[1]} features")
print(f"  🎯 Taxa de conversão: {(y == 1).sum() / len(y) * 100:.2f}%")

print(f"\n📊 Arquivos Disponíveis:")
print(f"  • data/processed/DATASETS_COMPARISON.csv")
print(f"  • data/processed/{SELECTED_DATASET}/data_processed.csv")
print(f"  • data/processed/{SELECTED_DATASET}/dataset_info.md")
print(f"  • models/scaler_{SELECTED_DATASET}.pkl")
print(f"  • models/label_encoders_{SELECTED_DATASET}.pkl")

print(f"\n💡 Para executar novamente:")
print(f"  Execute as células na ordem")
print(f"\n🚀 Próxima fase: Fase 2 - Baseline + Thompson Sampling")
print("="*70)


✅ FASE 1 CONCLUÍDA - RESUMO FINAL

📥 Downloads com Cache:
  ✅ 4/4 bases baixadas

📖 Datasets Carregados:
  ✅ 1️⃣ Bank Marketing (henriqueyamahata)
  ✅ 2️⃣ Bank Marketing Data Set (hariharanpavan)
  ✅ 3️⃣ Bank Term Deposit Subscription (dharmik34)
  ✅ 4️⃣ Telemarketing JYB Dataset (aguado)

🔄 Dataset Processado:
  📊 1️⃣ Bank Marketing (henriqueyamahata)
  📁 Salvo em: data/processed/bank-marketing/
  💾 41,188 × 20 features
  🎯 Taxa de conversão: 11.27%

📊 Arquivos Disponíveis:
  • data/processed/DATASETS_COMPARISON.csv (tabela de todos)
  • data/processed/bank-marketing/data_processed.csv
  • data/processed/bank-marketing/dataset_info.md
  • models/scaler_bank-marketing.pkl
  • models/label_encoders_bank-marketing.pkl

💡 Para processar outro dataset:
  1. Altere SELECTED_DATASET na célula 7️⃣
  2. Execute novamente as células 8️⃣-1️⃣2️⃣

🚀 Próxima fase: Fase 2 - Baseline + Thompson Sampling
